## Imports and Dependencies

This section imports all required libraries used for model definition, training, and evaluation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from pytorch_lightning import Trainer
import pytorch_lightning as pl
import torchmetrics
import timm
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import transforms,datasets
from pytorch_lightning.callbacks import EarlyStopping
import time
from torch.optim.lr_scheduler import StepLR

## Custom Rational Activation Function

This section defines a learnable rational activation function used within the GR-KAN architecture.

In [ ]:
class RationalFunction(nn.Module):
    """
    Implements the rational base function used in GR-KAN.
    The function has the form: F(x) = P(x) / (1 + |Q(x)|)
    where P and Q are polynomials.
    """
    def __init__(self, degree_num=5, degree_den=4, group_size=1):
        super().__init__()
        self.degree_num = degree_num
        self.degree_den = degree_den
        self.group_size = group_size
        
        # Coefficients for numerator (P(x)) and denominator (Q(x))
        self.a = nn.Parameter(torch.zeros(group_size, degree_num + 1))  # +1 for constant term
        self.b = nn.Parameter(torch.zeros(group_size, degree_den))      # no constant term in denominator
        
        self.initialize_coefficients()
        
    def initialize_coefficients(self):
        """Initialize coefficients to approximate identity function"""
        nn.init.constant_(self.a[:, 0], 0.0)  # constant term
        nn.init.constant_(self.a[:, 1], 1.0)  # linear term
        nn.init.constant_(self.a[:, 2:], 0.0)  # higher order terms
        
        # Initialize denominator to small values
        nn.init.normal_(self.b, mean=0.0, std=0.1)
        
    def forward(self, x):
        """
        Compute rational function using Horner's method for efficiency
        x: input tensor of shape (..., group_size)
        returns: output tensor of same shape as input
        """
        # Reshape for group processing
        orig_shape = x.shape
        x = x.view(-1, self.group_size)
        
        numerator = self.a[:, 0]
        for i in range(1, self.degree_num + 1):
            numerator = numerator + self.a[:, i] * (x ** i)
        
        denominator = torch.zeros_like(x)
        for i in range(1, self.degree_den + 1):
            denominator = denominator + self.b[:, i-1] * (x ** i)
        
        output = numerator / (1 + torch.abs(denominator))
        
        return output.view(*orig_shape)

## Model Definition

This section defines the neural network architecture used in the experiments.

In [ ]:
class KAT_Group(nn.Module):
    """
    Group-Rational KAN (GR-KAN) layer that replaces MLP in transformers.
    Implements the equation: GR-KAN(x) = W * F(x)
    where F is the group-wise rational function.
    """
    def __init__(self, dim_in, dim_out, num_groups=8, degree_num=5, degree_den=4):
        super().__init__()
        self.dim_in = dim_in
        self.dim_out = dim_out
        self.num_groups = num_groups
        self.group_size = dim_in // num_groups
        
        assert dim_in % num_groups == 0
        
        self.rational_fn = RationalFunction(degree_num, degree_den, num_groups)
        
        self.weight = nn.Parameter(torch.empty(dim_out, dim_in))
        self.bias = nn.Parameter(torch.empty(dim_out))
        
        self.reset_parameters()
        
    def reset_parameters(self):
        """Initialize weights using variance-preserving initialization"""
        gain = 1.0  # This should be adjusted based on actual rational function behavior
        
        nn.init.xavier_uniform_(self.weight, gain=math.sqrt(gain / self.dim_in))
        
        if self.bias is not None:
            nn.init.zeros_(self.bias)
            
    def forward(self, x):
        orig_shape = x.shape
        if len(orig_shape) == 3:
            batch_size, seq_len, dim_in = orig_shape
            x = x.reshape(batch_size * seq_len, dim_in)
        elif len(orig_shape) == 2:
            batch_size, dim_in = orig_shape
            seq_len = 1
        else:
            raise ValueError(f"Unexpected input shape: {orig_shape}")
        x = x.view(-1, self.num_groups, self.group_size)
        
        fx = self.rational_fn(x)
        
        fx = fx.view(*orig_shape)
        
        output = F.linear(fx, self.weight, self.bias)
        
        return output

    def dimensions(self):
        return f"dim_in={self.dim_in}, dim_out={self.dim_out}, num_groups={self.num_groups}"

In [ ]:
class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=2,
        spline_order=2,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=torch.nn.GELU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                # torch.nn.init.constant_(self.spline_scaler, self.scale_spline)
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        Compute the B-spline bases for the given input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).

        Returns:
            torch.Tensor: B-spline bases tensor of shape (batch_size, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )  # (in_features, grid_size + 2 * spline_order + 1)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        assert x.size(-1) == self.in_features
        original_shape = x.shape
        x = x.view(-1, self.in_features)

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        output = base_output + spline_output
        
        output = output.view(*original_shape[:-1], self.out_features)
        return output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin=0.01):
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  # (batch, in, coeff)
        splines = splines.permute(1, 0, 2)  # (in, batch, coeff)
        orig_coeff = self.scaled_spline_weight  # (out, in, coeff)
        orig_coeff = orig_coeff.permute(1, 2, 0)  # (in, coeff, out)
        unreduced_spline_output = torch.bmm(splines, orig_coeff)  # (in, batch, out)
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        )  # (batch, in, out)

        # sort each channel individually to collect data distribution
        x_sorted = torch.sort(x, dim=0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype=torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1),
            ],
            dim=0,
        )

        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """
        Compute the regularization loss.

        This is a dumb simulation of the original L1 regularization as stated in the
        paper, since the original one requires computing absolutes and entropy from the
        expanded (batch, in_features, out_features) intermediate tensor, which is hidden
        behind the F.linear function if we want an memory efficient implementation.

        The L1 regularization is now computed as mean absolute value of the spline
        weights. The authors implementation also includes this term in addition to the
        sample-based regularization.
        """
        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )

In [ ]:
class MyModel(pl.LightningModule):
    def __init__(self,model,lr=1e-3,weight_decay=1e-4,num_classes=3):
        super().__init__()
        self.model = model
        self.val_acc = torchmetrics.Accuracy(task="multiclass",num_classes=num_classes)
        self.train_acc = torchmetrics.Accuracy(task="multiclass",num_classes=num_classes)
        self.test_acc=torchmetrics.Accuracy(task="multiclass",num_classes=num_classes)
        self.criterion = torch.nn.CrossEntropyLoss()
        self.lr=lr
        self.weight_decay=weight_decay
    def forward(self, x):
        return self.model(x)
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.train_acc(preds, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_acc", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
        loss = self.criterion(logits, y)

        acc = self.val_acc(preds, y)
        self.log("val_acc", acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss
    def configure_optimizers(self):
        optimizer=torch.optim.AdamW(params=self.model.parameters(),lr=self.lr,weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,        # apply every epoch
                "monitor": "val_loss",
            }
        }
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, _ = batch
        logits = self(x)
        return torch.argmax(logits, dim=1)
    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.test_acc(preds, y)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test_acc", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

In [ ]:
def cross_validate(make_model,dataset, k=5, batch_size=32, num_workers=2, lr=1e-3, max_epochs=12,
                  weight_decay=1e-4,num_classes=3):
    kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    labels = [sample[1] for sample in dataset.samples]

    test_losses= []
    test_results=[]
    train_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(p=0.2),
        transforms.RandomRotation(0.2),
        transforms.RandAugment(num_ops=2,magnitude=9),
        transforms.ColorJitter(0.2,0.2,0.2,0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    test_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    for fold, (train_idx, test_idx) in enumerate(kfold.split(np.arange(len(labels)), labels)):
        print(f"\n===== Fold {fold+1}/{k} =====")
        train_val_indices = list(train_idx)
        split_point = int(0.8 * len(train_val_indices))
        train_indices = train_val_indices[:split_point]
        val_indices = train_val_indices[split_point:]

        train_subset = Subset(dataset, train_idx)
        #train_subset=Subset(dataset,train_indices)
        #val_subset=Subset(dataset,val_indices)
        test_subset = Subset(dataset, test_idx)

        train_dataset = TransformDataset(train_subset, transform=train_transforms)
        #val_dataset=TransformDataset(val_subset,transform=test_transforms)
        test_dataset = TransformDataset(test_subset, transform=test_transforms)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        #val_loader=DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        model=make_model()
        lightning_model = MyModel(model, lr=lr,weight_decay=weight_decay,num_classes=num_classes)
        #stop=EarlyStopping(monitor="val_loss",min_delta=1e-3,patience=5)

        trainer=Trainer(accelerator="gpu",max_epochs=max_epochs,logger=False)
        t0=time.time()
        train_metrics=trainer.fit(lightning_model, train_loader)
        timer=time.time()-t0
        test_metrics = trainer.test(lightning_model, dataloaders=test_loader)
        test_acc = test_metrics[0].get("test_acc", float("nan"))
        test_loss=test_metrics[0].get("test_loss",float("nan"))
        test_results.append(test_acc)
        test_losses.append(test_loss)
        print(f"Fold {fold+1}")
        del model,lightning_model, trainer
        torch.cuda.empty_cache()
    print("\n===== Final Results =====")
    print(f"Test Accuracies: {test_results}")
    print(f"Mean Test Acc: {np.mean(test_results):.4f} ± {np.std(test_results):.4f}")
    print("Mean Test Loss: {} ± {}".format(np.mean(test_losses),np.std(test_losses)))
    
    print("Model training time per fold:{} seconds".format(timer))
    return {"test_accs": test_results,"test_losses":test_losses}

In [ ]:
class TransformDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

In [ ]:
ld_dir="/kaggle/input/skyview-an-aerial-landscape-dataset/Aerial_Landscapes"
landscape_dataset=datasets.ImageFolder(root=ld_dir, transform=None)

In [ ]:
len(landscape_dataset)

## EDA
In this section we perform a small Exploratory Data Analysis on the dataset

In [ ]:
labels = np.array(landscape_dataset.targets)
class_names = landscape_dataset.classes

counts = np.bincount(labels)

dist = pd.DataFrame({
    "class": class_names,
    "count": counts
}).sort_values("count", ascending=False)

print(dist)

In [ ]:
from PIL import Image
from collections import Counter

sizes = []
for path, _ in landscape_dataset.samples:
    with Image.open(path) as img:
        sizes.append(img.size)

In [ ]:
df = pd.DataFrame(sizes, columns=["width", "height"])
df["aspect_ratio"] = df["width"] / df["height"]

print(df.describe())

## Fine-Tuning
In this section we train the specified model.

In [ ]:
def make_eva_model(num_blocks,layer_type,head=False,num_classes=15):
    eva=timm.create_model('eva02_small_patch14_224.mim_in22k',pretrained=True,num_classes=num_classes,img_size=256,global_pool="max",drop_rate=0.1,
                     pos_drop_rate=0.1,proj_drop_rate=0.1,attn_drop_rate=0.1,patch_drop_rate=0.1)
    layer_type.lower()
    if layer_type=="kan":
        LayerClass=KANLinear
    elif layer_type=="kat":
        LayerClass=KAT_Group
    if head:
        eva.head=LayerClass(384,num_classes)
    if num_blocks==0:
        return eva
    else:
        for i in range(11,11-num_blocks,-1):
            eva.blocks[i].mlp.fc1=LayerClass(384,2048)
            eva.blocks[i].mlp.fc2=LayerClass(1024,384)
        
    return eva

In [ ]:
example=cross_validate(lambda:make_eva_model(num_blocks=0,layer_typ="kan",head=True,num_classes=15),
                dataset=landscape_dataset,
                num_classes=15,
                lr=1e-4,
                weight_decay=1e-4,
                max_epochs=5)

## Visualization

In this section we try to visualize the learnt splines

In [ ]:
def plot_all_splines(layer, n_points=300, max_out=8):
    layer.eval()
    records = []

    for in_idx in range(8):
        grid = layer.grid[in_idx]
        xmin = grid[layer.spline_order].item()
        xmax = grid[-layer.spline_order - 1].item()

        x = torch.linspace(xmin, xmax, n_points, device=grid.device)

        for out_idx in range(min(layer.out_features, max_out)):
            y = eval_kan_spline(layer, x, in_idx, out_idx)

            records.append(
                pd.DataFrame({
                    "x": x.detach().cpu().numpy(),
                    "y": y.detach().cpu().numpy(),
                    "input": f"in_{in_idx}",
                    "output": f"out_{out_idx}"
                })
            )

    df = pd.concat(records, ignore_index=True)

    fig = px.line(
        df,
        x="x",
        y="y",
        color="output",
        facet_col="input",
        facet_col_wrap=4,
        title="EVA-KAN head learned splines"
    )

    fig.show()

In [ ]:
plot_all_splines(lightning_model.model.head)